# Phase 5 — Comparison metric

Computes `paper/comparison_metric.md`'s three layers (Plan Agreement Rate,
parametric agreement, outcome agreement) plus the degeneracy check and
reliability metrics, over Arm 1 (`results/rule_based_underserved.csv`,
deterministic, N=1 by construction) and Arm 2's two batches
(`results/llm_runs/`, canonical; `results/llm_runs_mitigated/`,
`system_hints`-mitigated).

**Runs entirely offline** — no network, no LLM key (`paper/PLAN.md`
Phase 5). Everything here reads only committed files; re-running this
notebook does not call the LLM again and does not change
`results/llm_runs*/`.

**Distance semantics, per `STUDY_LOG.md` — read before trusting a number
here.** Arm 1's headline undeserved counts (325 / 251 / 207 at 1000 / 1500
/ 2000 m) use **centroid** distance (`dist_centroid_m`). Every Arm 2 run in
every batch measures from the mahalle **polygon boundary** instead (the
`nearest_neighbor`/`spatial_nearest` op's default on polygon input) —
confirmed in the 0.5.5 and 0.5.6 batches by direct comparison, not assumed.
This notebook therefore compares Arm 2 against Arm 1's `dist_boundary_m`
column, never the headline centroid counts, and reports the
boundary-vs-centroid gap as its own line rather than silently reconciling
it.


## 1. Load Arm 1 (rule-based, N=1 by construction)

In [1]:
import json
import math
from itertools import combinations
from pathlib import Path

import pandas as pd
from scipy.stats import spearmanr

RESULTS = Path("../results")

arm1_df = pd.read_csv(RESULTS / "rule_based_underserved.csv")
arm1_df["osm_id"] = arm1_df["osm_id"].astype(int)
print(f"Arm 1: {len(arm1_df)} mahalle")
print(arm1_df[["dist_boundary_m", "dist_centroid_m"]].describe())

# Boundary-distance underserved sets at the same three tiers STUDY_LOG.md
# reports for the (centroid-based) headline numbers -- these are the
# correct comparison target for Arm 2, not the underserved_1000/1500/2000
# columns (those are centroid-based; see the markdown above).
ARM1_TIERS = (800, 1000, 1500, 2000)  # 800 included: one real Arm 2 run chose it
arm1_boundary_dist = dict(zip(arm1_df["osm_id"], arm1_df["dist_boundary_m"]))
arm1_centroid_dist = dict(zip(arm1_df["osm_id"], arm1_df["dist_centroid_m"]))
arm1_boundary_sets = {
    t: frozenset(oid for oid, d in arm1_boundary_dist.items() if d > t)
    for t in ARM1_TIERS
}
for t, s in arm1_boundary_sets.items():
    print(f"Arm 1 boundary-distance underserved at {t} m: {len(s)}")

# Headline (centroid) counts, reproduced here only to state the
# boundary-vs-centroid gap explicitly -- not used as a comparison target.
arm1_centroid_headline = {
    t: int((arm1_df["dist_centroid_m"] > t).sum()) for t in (1000, 1500, 2000)
}
print("\nFor reference, Arm 1's headline centroid-based counts:", arm1_centroid_headline)


Arm 1: 964 mahalle
       dist_boundary_m  dist_centroid_m
count       964.000000       964.000000
mean        830.651727      1762.381339
std        1922.744376      2703.302055
min           0.000000        20.325823
25%           0.000000       333.161810
50%          20.097468       610.054189
75%         440.207875      1624.616578
max       13895.635278     17294.618111
Arm 1 boundary-distance underserved at 800 m: 183
Arm 1 boundary-distance underserved at 1000 m: 166
Arm 1 boundary-distance underserved at 1500 m: 142
Arm 1 boundary-distance underserved at 2000 m: 127

For reference, Arm 1's headline centroid-based counts: {1000: 325, 1500: 251, 2000: 207}


## 2. Load Arm 2 batches from raw `run_*.json`

Reads every saved record verbatim — success or failure, same discipline
as `03_llm_arm.ipynb`. Extracts, per successful run: the operation
sequence (Layer 1), the chosen CRS / `max_distance` / filter threshold
(Layer 2), and the output feature set with its distance values (Layer 3).

**Known limitation, stated here rather than glossed over**: `s3geo.query()`
returns a single `result.output` — the *last* declared output
(`underserved_mahalle`), even though every plan's `QuerySpec.outputs` also
names an earlier, unfiltered per-mahalle distance node
(`mahalle_with_distance` / `nearest_hospital`). That earlier node is
computed during execution but never surfaced by the one-call API, so no
saved run file has a real per-mahalle distance for the ~780-800 mahalle
*not* in the underserved set. Layer 3b (Rank Stability) below is therefore
computed only over the intersection of each pair's *output* sets, not the
full 964 mahalle `paper/comparison_metric.md` specifies — flagged as a
real gap in "Limitations" below, not silently narrowed.

In [2]:
def find_ops(operations, names):
    return [o for o in operations if o["op"] in names]


def load_run(path):
    return json.loads(path.read_text(encoding="utf-8"))


def extract(record):
    """One run_*.json record -> the fields every layer of the metric needs.
    Returns None fields throughout on failure -- a failure is data, not
    something to paper over (STUDY_LOG.md)."""
    out = {
        "success": record["success"],
        "error_stage": record.get("error_stage"),
        "attempt_count": record.get("attempt_count"),
        "repaired": record.get("repaired"),
        "latency_s": record.get("latency_s"),
        "operations": tuple(record["operations"]) if record.get("operations") else None,
        "crs_values": None,
        "max_distance": None,
        "threshold": None,
        "where_op": None,
        "distance_field": None,
        "osm_ids": None,
        "distances": None,
        "n_output": None,
    }
    if not record["success"]:
        return out

    qs = record["query_spec"]
    ops = qs["operations"]
    crs_ops = find_ops(ops, {"crs_transform"})
    out["crs_values"] = tuple(sorted({o["params"].get("target_crs") for o in crs_ops}))

    nn_ops = find_ops(ops, {"spatial_nearest", "nearest_neighbor"})
    distance_field = "_nearest_distance"
    if nn_ops:
        out["max_distance"] = nn_ops[0]["params"].get("max_distance")
        distance_field = nn_ops[0]["params"].get("distance_field") or "_nearest_distance"
    out["distance_field"] = distance_field

    filt_ops = find_ops(ops, {"filter_attribute"})
    if filt_ops:
        where = filt_ops[0]["params"].get("where") or {}
        out["threshold"] = where.get("value")
        out["where_op"] = where.get("op")

    feats = (record["output"] or {}).get("features", [])
    osm_ids, dists = [], {}
    for feat in feats:
        props = feat.get("properties", {})
        oid = props.get("osm_id")
        dist = props.get(distance_field)
        if oid is not None:
            osm_ids.append(int(oid))
            if isinstance(dist, (int, float)):
                dists[int(oid)] = dist
    out["osm_ids"] = frozenset(osm_ids)
    out["distances"] = dists
    out["n_output"] = len(feats)
    return out


def load_batch(dir_name, n_runs):
    rows = []
    d = RESULTS / dir_name
    for i in range(n_runs):
        p = d / f"run_{i:02d}.json"
        if not p.exists():
            continue
        rows.append({"run_index": i, **extract(load_run(p))})
    return rows


canonical = load_batch("llm_runs", 20)
mitigated = load_batch("llm_runs_mitigated", 20)

print(f"canonical: {len(canonical)} records, {sum(r['success'] for r in canonical)} successful")
print(f"mitigated: {len(mitigated)} records, {sum(r['success'] for r in mitigated)} successful")


canonical: 20 records, 20 successful
mitigated: 20 records, 18 successful


## 3. Degeneracy check (before any metric — STUDY_LOG.md / comparison_metric.md)

A run is degenerate if its output set is empty, is everything (964), or
every kept feature has the identical distance value. Recorded as its own
column, separate from `success` — a batch can be 100% "successful" and
100% degenerate at the same time (this happened in earlier Vienna
batches, per `comparison_metric.md`).

In [3]:
def is_degenerate(row):
    if not row["success"]:
        return None  # not applicable -- didn't execute at all
    n = row["n_output"]
    if n == 0 or n == len(arm1_df):
        return True
    vals = set(row["distances"].values())
    if n > 1 and len(vals) <= 1:
        return True
    return False


for batch in (canonical, mitigated):
    for row in batch:
        row["degenerate"] = is_degenerate(row)

print("canonical degenerate count:", sum(1 for r in canonical if r["degenerate"]))
print("mitigated degenerate count:", sum(1 for r in mitigated if r["degenerate"]))


canonical degenerate count: 0
mitigated degenerate count: 0


## 4. Layer 1 — Plan Agreement Rate (PAR)

In [4]:
def par(batch):
    seqs = [r["operations"] for r in batch if r["success"]]
    if not seqs:
        return 0.0, None, 0
    mode_seq = max(set(seqs), key=seqs.count)
    agree = sum(1 for s in seqs if s == mode_seq)
    # Per comparison_metric.md, PAR is over all N runs (a structural
    # failure -- no plan at all -- counts against it, same denominator
    # `success_rate` uses below), not only over the successful ones.
    return agree / len(batch), mode_seq, agree


for name, batch in (("canonical", canonical), ("mitigated", mitigated)):
    rate, mode_seq, agree = par(batch)
    print(f"{name}: PAR = {agree}/{len(batch)} = {rate:.3f}  mode sequence = {mode_seq}")

print("\nArm 1 (rule-based): PAR = 1.0 by construction (comparison_metric.md).")


canonical: PAR = 20/20 = 1.000  mode sequence = ('crs_transform', 'crs_transform', 'spatial_nearest', 'filter_attribute')
mitigated: PAR = 18/20 = 0.900  mode sequence = ('crs_transform', 'crs_transform', 'spatial_nearest', 'filter_attribute')

Arm 1 (rule-based): PAR = 1.0 by construction (comparison_metric.md).


## 5. Layer 2 — Parametric agreement

Among runs sharing the mode sequence: threshold (mean ± sd), CRS
(should be zero-variance — flagged separately if not), and
`max_distance` (flagged separately per `comparison_metric.md`: "a plan
that picks a geographic CRS ... is a wrong answer, and it should be
counted separately rather than averaged in" — the same treatment applies
to a self-defeating `max_distance`).

In [5]:
def parametric_summary(batch, mode_seq):
    rows = [r for r in batch if r["success"] and r["operations"] == mode_seq]
    thresholds = [r["threshold"] for r in rows if isinstance(r["threshold"], (int, float))]
    crs_sets = {r["crs_values"] for r in rows}
    max_dist_used = [r for r in rows if r["max_distance"] is not None]
    mean_t = sum(thresholds) / len(thresholds) if thresholds else None
    sd_t = (
        (sum((t - mean_t) ** 2 for t in thresholds) / len(thresholds)) ** 0.5
        if thresholds else None
    )
    return {
        "n_in_mode": len(rows),
        "threshold_mean": mean_t,
        "threshold_sd": sd_t,
        "threshold_values": sorted(thresholds),
        "distinct_crs_sets": crs_sets,
        "n_using_max_distance": len(max_dist_used),
    }


param_summaries = {}
for name, batch in (("canonical", canonical), ("mitigated", mitigated)):
    _, mode_seq, _ = par(batch)
    s = parametric_summary(batch, mode_seq)
    param_summaries[name] = s
    print(f"\n{name}:")
    print(f"  threshold: mean={s['threshold_mean']:.1f}  sd={s['threshold_sd']:.1f}  values={s['threshold_values']}")
    print(f"  CRS used (should be one set, zero variance): {s['distinct_crs_sets']}")
    print(f"  runs with a max_distance cap set at all (flag, don't average in): {s['n_using_max_distance']} / {s['n_in_mode']}")



canonical:
  threshold: mean=990.0  sd=43.6  values=[800, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000]
  CRS used (should be one set, zero variance): {('EPSG:32635',)}
  runs with a max_distance cap set at all (flag, don't average in): 0 / 20

mitigated:
  threshold: mean=1000.0  sd=0.0  values=[1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000]
  CRS used (should be one set, zero variance): {('EPSG:32635',)}
  runs with a max_distance cap set at all (flag, don't average in): 0 / 18


## 6. Layer 3a — Outcome agreement: set overlap (Jaccard)

Pairwise Jaccard within each batch (Set Stability), and Jaccard of every
successful run against Arm 1's **boundary**-distance set at that run's own
chosen threshold (never the centroid headline — see §1).

In [6]:
def jaccard(a, b):
    u = a | b
    return len(a & b) / len(u) if u else 1.0


def set_stability(batch):
    succ = [r for r in batch if r["success"]]
    pairs = list(combinations(succ, 2))
    if not pairs:
        return None, None, 0
    js = [jaccard(a["osm_ids"], b["osm_ids"]) for a, b in pairs]
    mean_j = sum(js) / len(js)
    sd_j = (sum((j - mean_j) ** 2 for j in js) / len(js)) ** 0.5
    return mean_j, sd_j, len(pairs)


def vs_arm1(row):
    t = row["threshold"]
    if t is None or t not in arm1_boundary_sets:
        # threshold not one of the pre-computed tiers -- compute on the fly
        ref = frozenset(oid for oid, d in arm1_boundary_dist.items() if d > t) if t is not None else None
    else:
        ref = arm1_boundary_sets[t]
    if ref is None:
        return None, None
    j = jaccard(row["osm_ids"], ref)
    return j, (row["osm_ids"] == ref)


for name, batch in (("canonical", canonical), ("mitigated", mitigated)):
    mean_j, sd_j, n_pairs = set_stability(batch)
    print(f"\n{name}: Set Stability (mean pairwise Jaccard, {n_pairs} pairs) = {mean_j:.4f} (sd={sd_j:.4f})" if mean_j is not None else f"\n{name}: no successful pairs")
    exact_count = 0
    j_values = []
    for r in batch:
        if not r["success"]:
            continue
        j, exact = vs_arm1(r)
        r["jaccard_vs_arm1"] = j
        r["exact_match_vs_arm1"] = exact
        if j is not None:
            j_values.append(j)
        if exact:
            exact_count += 1
    n_succ = sum(1 for r in batch if r["success"])
    print(f"{name}: Jaccard vs. Arm 1 (own threshold) mean={sum(j_values)/len(j_values):.4f}, "
          f"{exact_count}/{n_succ} successful runs are an EXACT set match to Arm 1's boundary set at their own threshold")

print("\nArm 1 vs. itself: Jaccard = 1.0, Set Stability = 1.0 by construction.")



canonical: Set Stability (mean pairwise Jaccard, 190 pairs) = 0.9907 (sd=0.0279)
canonical: Jaccard vs. Arm 1 (own threshold) mean=1.0000, 20/20 successful runs are an EXACT set match to Arm 1's boundary set at their own threshold

mitigated: Set Stability (mean pairwise Jaccard, 153 pairs) = 1.0000 (sd=0.0000)
mitigated: Jaccard vs. Arm 1 (own threshold) mean=1.0000, 18/18 successful runs are an EXACT set match to Arm 1's boundary set at their own threshold

Arm 1 vs. itself: Jaccard = 1.0, Set Stability = 1.0 by construction.


## 7. Layer 3b — Outcome agreement: rank stability (Spearman ρ)

**Partial, per the §2 limitation**: computed only over each pair's shared
output osm_ids (the mahalle both runs happened to flag as underserved),
not the full 964 — `s3geo.query()` never surfaces the unfiltered
per-mahalle distance. Reported as a lower bound / sanity check, not the
full-population statistic `comparison_metric.md` specifies.

In [7]:
def rank_stability_partial(batch):
    succ = [r for r in batch if r["success"] and r["distances"]]
    pairs = list(combinations(succ, 2))
    rhos = []
    for a, b in pairs:
        common = sorted(set(a["distances"]) & set(b["distances"]))
        if len(common) < 3:
            continue
        xa = [a["distances"][k] for k in common]
        xb = [b["distances"][k] for k in common]
        rho, _ = spearmanr(xa, xb)
        if not math.isnan(rho):
            rhos.append(rho)
    return (sum(rhos) / len(rhos) if rhos else None), len(rhos)


for name, batch in (("canonical", canonical), ("mitigated", mitigated)):
    mean_rho, n = rank_stability_partial(batch)
    print(f"{name}: partial Rank Stability (mean ρ over {n} pairs, shared-output-only) = "
          f"{mean_rho:.4f}" if mean_rho is not None else f"{name}: not enough overlapping data")

print("\nArm 1 vs. itself: ρ = 1.0 by construction.")


canonical: partial Rank Stability (mean ρ over 190 pairs, shared-output-only) = 1.0000
mitigated: partial Rank Stability (mean ρ over 153 pairs, shared-output-only) = 1.0000

Arm 1 vs. itself: ρ = 1.0 by construction.


## 8. Reliability metrics

In [8]:
def reliability(batch):
    n = len(batch)
    n_success = sum(1 for r in batch if r["success"])
    n_degenerate = sum(1 for r in batch if r.get("degenerate"))
    latencies = sorted(r["latency_s"] for r in batch if r["latency_s"] is not None)
    n_repaired = sum(1 for r in batch if r.get("repaired"))
    n_attempted_repair_and_failed = sum(
        1 for r in batch if not r["success"] and (r.get("attempt_count") or 0) >= 1
    )

    def median(xs):
        m = len(xs)
        if m == 0:
            return None
        mid = m // 2
        return xs[mid] if m % 2 else (xs[mid - 1] + xs[mid]) / 2

    def iqr(xs):
        if len(xs) < 4:
            return None
        q1 = xs[len(xs) // 4]
        q3 = xs[(3 * len(xs)) // 4]
        return q3 - q1

    return {
        "n": n,
        "success_rate": n_success / n,
        "degenerate_rate": n_degenerate / n_success if n_success else None,
        "median_latency_s": median(latencies),
        "latency_iqr_s": iqr(latencies),
        "n_repaired": n_repaired,
        "n_generation_errors_no_repair_available": sum(
            1 for r in batch if not r["success"] and (r.get("attempt_count") or 0) == 0
        ),
    }


rel = {}
for name, batch in (("canonical", canonical), ("mitigated", mitigated)):
    rel[name] = reliability(batch)
    print(name, json.dumps(rel[name], indent=2))


canonical {
  "n": 20,
  "success_rate": 1.0,
  "degenerate_rate": 0.0,
  "median_latency_s": 14.403500000000001,
  "latency_iqr_s": 2.058,
  "n_repaired": 0,
  "n_generation_errors_no_repair_available": 0
}
mitigated {
  "n": 20,
  "success_rate": 0.9,
  "degenerate_rate": 0.0,
  "median_latency_s": 14.3035,
  "latency_iqr_s": 0.8000000000000007,
  "n_repaired": 0,
  "n_generation_errors_no_repair_available": 2
}


## 9. Summary table (`comparison_metric.md` "Reporting format")

One row per arm/batch. Arm 1 is deterministic (N=1); its stability columns
are `1.0` by construction, not computed.

In [9]:
def build_row(name, batch, threshold_mean, threshold_sd, set_stab, jacc_vs_arm1_mean, mean_u, rank_stab, rel_row):
    return {
        "Arm": name,
        "N": rel_row["n"],
        "PAR": round(par(batch)[0], 3),
        "Threshold (mean±sd)": f"{threshold_mean:.0f} ± {threshold_sd:.0f}" if threshold_mean is not None else "n/a",
        "Set Stability (Jaccard)": f"{set_stab[0]:.4f} ± {set_stab[1]:.4f}" if set_stab[0] is not None else "n/a",
        "Jaccard vs. rule-based": f"{jacc_vs_arm1_mean:.4f}" if jacc_vs_arm1_mean is not None else "n/a",
        "|U| (mean±sd)": mean_u,
        "Rank Stability (ρ, partial)": f"{rank_stab:.4f}" if rank_stab is not None else "n/a",
        "Success": f"{rel_row['success_rate']:.2f}",
        "Degenerate": f"{rel_row['degenerate_rate']:.2f}" if rel_row["degenerate_rate"] is not None else "n/a",
        "Median latency (s)": f"{rel_row['median_latency_s']:.1f}" if rel_row["median_latency_s"] is not None else "n/a",
    }


rows = []
rows.append({
    "Arm": "rule-based (Arm 1)", "N": 1, "PAR": 1.0,
    "Threshold (mean±sd)": "2000 ± 0 (by design; 1000/1500 also reported)",
    "Set Stability (Jaccard)": "1.0 (by construction)",
    "Jaccard vs. rule-based": "1.0",
    "|U| (mean±sd)": f"{arm1_centroid_headline[2000]} (centroid, 2000 m headline)",
    "Rank Stability (ρ, partial)": "1.0 (by construction)",
    "Success": "1.00", "Degenerate": "0.00", "Median latency (s)": "n/a (deterministic)",
})

for name, batch in (("LLM, canonical (Arm 2)", canonical), ("LLM, mitigated (Arm 2)", mitigated)):
    key = "canonical" if "canonical" in name else "mitigated"
    s = param_summaries[key]
    mean_j, sd_j, _ = set_stability(batch)
    j_vals = [r["jaccard_vs_arm1"] for r in batch if r.get("jaccard_vs_arm1") is not None]
    u_vals = [r["n_output"] for r in batch if r["success"]]
    mean_u = sum(u_vals) / len(u_vals) if u_vals else None
    sd_u = (sum((u - mean_u) ** 2 for u in u_vals) / len(u_vals)) ** 0.5 if u_vals else None
    rank_stab, _ = rank_stability_partial(batch)
    rows.append(build_row(
        name, batch, s["threshold_mean"], s["threshold_sd"],
        (mean_j, sd_j), (sum(j_vals) / len(j_vals) if j_vals else None),
        f"{mean_u:.1f} ± {sd_u:.1f}" if mean_u is not None else "n/a",
        rank_stab, rel[key],
    ))

summary_df = pd.DataFrame(rows)
summary_df


,Arm,N,PAR,Threshold (mean±sd),Set Stability (Jaccard),Jaccard vs. rule-based,|U| (mean±sd),"Rank Stability (ρ, partial)",Success,Degenerate,Median latency (s)
0,rule-based (Arm 1),1,1.0,2000 ± 0 (by design; 1000/1500 also reported),1.0 (by construction),1.0,"207 (centroid, 2000 m headline)",1.0 (by construction),1.00,0.00,n/a (deterministic)
1,"LLM, canonical (Arm 2)",20,1.0,990 ± 44,0.9907 ± 0.0279,1.0000,166.8 ± 3.7,1.0000,1.00,0.00,14.4
2,"LLM, mitigated (Arm 2)",20,0.9,1000 ± 0,1.0000 ± 0.0000,1.0000,166.0 ± 0.0,1.0000,0.90,0.00,14.3


In [10]:
summary_df.to_csv(RESULTS / "metrics.csv", index=False)
print(f"wrote {RESULTS / 'metrics.csv'}")


wrote ../results/metrics.csv


## 10. Limitations found while building this notebook

- **Rank Stability (§7) is partial, not the full-population statistic
  `comparison_metric.md` specifies.** `s3geo.query()`'s single
  `result.output` only ever returns the *last* declared output
  (`underserved_mahalle`), never the earlier, unfiltered per-mahalle
  distance node every plan also declares (`mahalle_with_distance` /
  `nearest_hospital`). Fixing this for a future batch needs either (a) a
  QuerySpec that declares only the unfiltered node as its output, run as a
  *second*, separate N=20 batch purely to capture full distances (doubling
  the LLM-call cost), or (b) reaching into `orchestrator.planning`
  directly (as `s3geo` itself does internally) to capture every
  intermediate DAG node's output rather than using the one-call wrapper.
  Neither is implemented here; flagging it is preferred to silently
  reporting a partial number as the full one.
- **Boundary vs. centroid distance is a real, load-bearing definitional
  gap, not a rounding issue.** Every Arm 2 run in every batch on record
  measures from the mahalle polygon boundary; Arm 1's headline numbers
  (`STUDY_LOG.md`, `paper/PLAN.md`) use the centroid. This notebook compares
  Arm 2 only against Arm 1's `dist_boundary_m` column for that reason —
  see §1.
- **This notebook does not call the LLM and does not know why a run chose
  the threshold or CRS it did** — only what it chose. Reasoning about
  *why* (e.g. why 19/20 canonical runs picked exactly 1000 m) belongs in
  the paper's discussion, informed by `paper/PLAN.md`'s "Findings", not
  here.
